In [3]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
# 1. Read data from silver
df_pro_silver = spark.read.table("silver_DimProduct")
df_cus_silver = spark.read.table("silver_DimCustomer")
df_geo_silver = spark.read.table("silver_DimGeography")
df_fact_silver = spark.read.table("silver_FactInternetSales")

# 2. Dimension
# Gold Product
gold_dim_product = df_pro_silver.select(
    "ProductKey", "ProductName", "CategoryName", "SubCategoryName", 
    "Color", "ListPrice", "Status"
)

# Gold Customer
gold_dim_customer = df_cus_silver.join(df_geo_silver, "GeographyKey", "left") \
                                .select("CustomerKey", "FullName", "Gender","MaritalStatus","TotalChildren","Education", "YearlyIncome", "Occupation", "City", "State", "Country")

# Gold Date
start_date = "2010-01-01"
end_date = "2025-12-31"
df_date = spark.sql(f"SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) as FullDateLabel")

gold_dim_date = df_date.select(
                                    F.date_format("FullDateLabel", "yyyyMMdd").cast("int").alias("DateKey"),
                                    F.col("FullDateLabel").alias("Date"),
                                    F.year("FullDateLabel").alias("Year"),
                                    F.quarter("FullDateLabel").alias("Quarter"),
                                    F.month("FullDateLabel").alias("Month"),
                                    F.date_format("FullDateLabel", "MMMM").alias("MonthName"),
                                    F.dayofweek("FullDateLabel").alias("DayOfWeek"),
                                    F.date_format("FullDateLabel", "EEEE").alias("DayName")
                                )

# 3. Fact Internetsales
gold_fact_sales = df_fact_silver.select(
    "SalesOrderNumber", "SalesOrderLineNumber", "ProductKey", "CustomerKey", "OrderDate", 
    "OrderQuantity", "UnitPrice", "SalesAmount", "NetRevenue"
)

if not spark.catalog.tableExists("gold_FactInternetSales"):
    # tạo mới nếu chưa tồn tại
    gold_fact_sales.write.format("delta").mode("overwrite").saveAsTable("gold_FactInternetSales")
else:
    # nạp bù nếu đã tồn tại
    target_table = DeltaTable.forName(spark, "gold_FactInternetSales")
    
    # định danh dòng: Số hóa đơn
    join_condition = "target.SalesOrderNumber = source.SalesOrderNumber AND target.SalesOrderLineNumber = source.SalesOrderLineNumber"
    
    target_table.alias("target").merge(gold_fact_sales.alias("source"),join_condition)\
                                .whenMatchedUpdateAll()\
                                .whenNotMatchedInsertAll()\
                                .execute()

# 4. Write
gold_dim_product.write.format("delta").mode("overwrite").saveAsTable("gold_DimProduct")
gold_dim_customer.write.format("delta").mode("overwrite").option('overwriteSchema','true').saveAsTable("gold_DimCustomer")
gold_dim_date.write.format("delta").mode("overwrite").saveAsTable("gold_DimDate")

StatementMeta(, a43a91ec-faf0-411f-b9f1-57d5894ceed2, 5, Finished, Available, Finished, False)